In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker
from scipy import stats
from statsmodels.stats.multitest import multipletests
import warnings

# Suppress warnings for clean output
warnings.filterwarnings('ignore')

# =============================================================================
# 1. Publication-Quality Visualization Settings (Nature Portfolio Standard)
# =============================================================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
DPI_SETTING = 600

# =============================================================================
# 2. Robust Data Loading & Strict Exclusion Rules (Zero-Imputation Avoidance)
# =============================================================================
def clean_and_convert_to_nan(df, cols):
    """
    Strictly converts non-numeric values to np.nan.
    Maintains scientific integrity by avoiding arbitrary zero-imputation.
    """
    for col in cols:
        df[col] = df[col].astype(str).replace(['Undetermined', '-', 'nan', '#VALUE!', ''], np.nan)
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

# Load raw datasets
df_ph = pd.read_csv('pH.csv')
df_buty = pd.read_csv('Butyrate(mM).csv')

donor_cols = [c for c in df_ph.columns if c.startswith('HS-')]

# Apply data cleaning
df_ph = clean_and_convert_to_nan(df_ph, donor_cols)
df_buty = clean_and_convert_to_nan(df_buty, donor_cols)

# =============================================================================
# 3. Calculate Delta (Inulin - Control)
# =============================================================================
ctrl_ph = df_ph[df_ph['KULFFI'].str.strip() == 'Control'][donor_cols].iloc[0]
ctrl_buty = df_buty[df_buty['KULFFI'].str.strip() == 'Control'][donor_cols].iloc[0]

inulin_ph = df_ph[df_ph['KULFFI'].str.strip() == 'Inulin'][donor_cols].iloc[0]
inulin_buty = df_buty[df_buty['KULFFI'].str.strip() == 'Inulin'][donor_cols].iloc[0]

delta_ph = inulin_ph - ctrl_ph
delta_buty = inulin_buty - ctrl_buty

df_plot = pd.DataFrame({
    'Delta_pH': delta_ph,
    'Delta_Butyrate': delta_buty
})

# Enforce strict pairwise exclusion for valid correlation plotting
df_plot = df_plot.dropna(subset=['Delta_pH', 'Delta_Butyrate'])

# =============================================================================
# 4. Ecological Stratification (Tertile Thresholds as per Manuscript)
# =============================================================================
def classify_ecotype(val):
    if val <= -0.80:
        return 'Severe'
    elif -0.80 < val < -0.44:
        return 'Moderate'
    else:
        return 'Mild'

# Apply stratification
df_plot['Ecotype'] = df_plot['Delta_pH'].apply(classify_ecotype)
# Define strict order for plotting (Top to Bottom: Mild -> Moderate -> Severe)
ecotype_order = ['Mild', 'Moderate', 'Severe']
df_plot['Ecotype'] = pd.Categorical(df_plot['Ecotype'], categories=ecotype_order, ordered=True)

# =============================================================================
# 5. Plotting Figure 6d (pH-driven Metabolic Stall)
# =============================================================================
fig, ax = plt.subplots(figsize=(6, 5))

# Colors defined in manuscript: Severe (blue), Moderate (grey), Mild (red)
palette = {'Mild': '#d62728', 'Moderate': '#7f7f7f', 'Severe': '#1f77b4'}

# Reference lines for baseline and ecological tipping points (Z-order: 0, bottom)
ax.axhline(0, color='black', linestyle=':', linewidth=1.5, zorder=0)
ax.axvline(-0.80, color=palette['Severe'], linestyle='--', linewidth=1.5, alpha=0.6, zorder=0)
ax.axvline(-0.44, color=palette['Mild'], linestyle='--', linewidth=1.5, alpha=0.6, zorder=0)

# Plot regression line (Z-order: 1, behind the points)
sns.regplot(x='Delta_pH', y='Delta_Butyrate', data=df_plot, scatter=False,
            color='darkgrey', line_kws={'linestyle': '-', 'linewidth': 2, 'zorder': 1}, ax=ax)

# Plot scatter points (Z-order: 3, top priority)
sns.scatterplot(x='Delta_pH', y='Delta_Butyrate', hue='Ecotype', data=df_plot,
                palette=palette, s=120, edgecolor='black', linewidth=1.0, alpha=0.9, zorder=3, ax=ax)

# X-axis ticks every 0.2 increments
ax.xaxis.set_major_locator(ticker.MultipleLocator(0.2))

# Axis Labels
ax.set_xlabel(r'$\Delta\,$pH', fontsize=16, fontweight='bold', labelpad=10)
ax.set_ylabel(r'$\Delta\,$Butyrate (mM)', fontsize=16, fontweight='bold', labelpad=10)

# Slightly reduced tick label size to prevent crowding on the x-axis
ax.tick_params(axis='both', labelsize=12, width=1.5, length=6)

for spine in ['left', 'bottom']:
    ax.spines[spine].set_linewidth(2.0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add Statistical Annotation (Pearson r and q-value) at Top-Left (No box, No background)
r, p = stats.pearsonr(df_plot['Delta_pH'], df_plot['Delta_Butyrate'])
# Calculate FDR corrected q-value (In this specific singular test context, it mirrors p, but formally processed)
_, q_vals, _, _ = multipletests([p], method='fdr_bh')
q = q_vals[0]

# Format q-value display (e.g., q < 0.001 or scientific notation)
q_text = "q < 0.001" if q < 0.001 else f"q = {q:.1e}"
ax.text(0.05, 0.95, f"$r = {r:.2f}$\n${q_text}$", transform=ax.transAxes,
        fontsize=14, fontweight='bold', va='top', ha='left', color='black')

# Format Legend (Ordered Mild -> Moderate -> Severe, outside right)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=handles, labels=labels, title='Ecotype (Inulin)',
          title_fontsize=12, fontsize=11, frameon=False,
          loc='upper left', bbox_to_anchor=(1.02, 1))

# Export
output_file = 'Figure_6f.pdf'
plt.savefig(output_file, dpi=DPI_SETTING, bbox_inches='tight')
print(f"Figure successfully saved as {output_file}")